# Predictions

Here we finally deploy our trained models to make predictions on real images.
The program uses Sliding Window method to extract sub-images from the full satellite image.
It then extracts features of those sub-images to determine the presence of waste in them.

In [1]:
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import pickle
import time
import Utilities

In [2]:
src_folder_path = '/Users/sinner/Desktop/Projects/SatellightSight - Waste detection using Satellite Images/All Folders/images to scan 2'

dest_folder_path = '/Users/sinner/Desktop/Annotated_Outputs'

In [3]:
# Defining a custom standard scaler to scale features

mean_std_df = pd.read_csv('mean_std_df.csv')

def my_standard_scaler(features):
    scaled_features = []
    for i, value in enumerate(features):
        mean = mean_std_df.iat[i, 1]
        std = mean_std_df.iat[i, 2]
        scaled_features.append((value - mean) / std)
    return np.array(scaled_features)

In [4]:
# Load model
loaded_model = pickle.load(open('trained_models/RandomForestClassifier.sav', 'rb'))
print('Model Loaded')

Model Loaded


In [5]:
# Driver code

start_time = time.time()
total_num_windows = 0

# iterating in source folder
for filename in os.listdir(src_folder_path):

    # Check whether filetype is correct
    if not filename.endswith(('.jpg', '.png', '.jpeg')):
        continue

    print(f"Scanning {filename}")

    # load image
    img_bgr = cv2.imread(os.path.join(src_folder_path, filename))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    positiveWindows = []
    for window_coord in Utilities.get_window_coords(img_rgb, 150, 150):
        total_num_windows += 1

        # get cropped window from img
        # cropped_img = img[top:bottom, left:right]
        window = img_rgb[round(window_coord[0]):round(window_coord[1]), round(window_coord[2]):round(window_coord[3])]

        # get features of window
        features = Utilities.get_features(window)
        # Feature Scaling
        features = my_standard_scaler(features)

        # Predict class using model
        predicted_class = loaded_model.predict(np.reshape(features, (1,-1)))
        # print(predicted_class)

        if predicted_class[0] == 1:
            # append coordinates into positiveWindows list if waste detected
            positiveWindows.append(window_coord)

        overlay_time_start = time.time()
        for window_coord in positiveWindows:
            # Overlay the bounding box on the image
            cv2.rectangle(img_bgr, (window_coord[2], window_coord[0]), (window_coord[3], window_coord[1]), (0, 255, 0), 2)

    cv2.imwrite(os.path.join(dest_folder_path, filename), img_bgr)

total_time = time.time() - start_time

print(f"Total time taken: {total_time}")
print(f"Average time per image = {total_time/len(os.listdir(src_folder_path))}")
print(f"Total number of windows scanned = {total_num_windows}")
print(f"Average time per window = {total_time/total_num_windows}")

Scanning 0714.jpg
Scanning 0715.jpg
Scanning 0711.jpg
Scanning 0710.jpg
Scanning 0712.jpg
Scanning 0706.jpg
Scanning 0707.jpg
Scanning 0713.jpg
Scanning 0709.jpg
Scanning 0708.jpg
Total time taken: 89.59135413169861
Average time per image = 8.95913541316986
Total number of windows = 4650
Average time per window = 0.01926695787778465


In [5]:
# Sliding window method Performance For 10 Images